In [1]:
import json
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt

In [2]:

def exact_match_accuracy(ground_truth_list, predictions, selector):
        """Calculate predicate + qualifier match accuracy in 3 levels."""

        if len(predictions) != len(ground_truth_list):
            raise ValueError("Predictions and ground truth must have same length")
 
        predicate_matches = 0
        mismatches = []
        total = len(predictions)
        for i, gt in enumerate(ground_truth_list):
            
            if not gt:
                print("whyyyyyyyy")
                continue
        
            pred = predictions[i]
            top = pred.get('top_choice')
            if selector and top.get('selector', '') != selector:
                print("whyyyyyyyy no sel")
                total-=1
                continue
            
            
            pred_pred = top.get('predicate', '').replace('biolink:', '')
            if isinstance(gt, str):
                if pred_pred == gt.lower():
                    predicate_matches += 1
                else:
                    mismatches.append(i)
            elif isinstance(gt, list):
                if pred_pred in gt:
                    predicate_matches += 1
                else:
                    mismatches.append(i)
        print("total instances evaluated for exact match:", total)
        return predicate_matches / total if total > 0 else 0.0, mismatches

def top_k_accuracy(ground_truth_list, predictions, k_values= [1, 3, 5, 10]):
        """Calculate top-k accuracy for predicates"""
        results = {}
    
        for k in k_values:
            correct = 0
            total = len(predictions)

            for i, pred in enumerate(predictions):
                gt_predicate = ground_truth_list[i]
                    
                if not gt_predicate: 
                    total -= 1
                    print(f"what!!!! in hit @{k}")
                    continue
                
                # top-k candidates
                candidates = []
                if 'Top_n_candidates' in pred:
                    for j in range(min(k, len(pred['Top_n_candidates']))):
                        # print(i, pred['Top_n_candidates'].keys())
                        if str(j) in pred['Top_n_candidates']:
                            candidate = pred['Top_n_candidates'][str(j)]['mapped_predicate']
                            candidates.append(candidate)
                else:
                    print("what!!!!")
                
                if isinstance(gt_predicate, str):
                    gt_predicate = {gt_predicate}
                else:
                    gt_predicate = set(gt_predicate)
                #  ground truth is in top-k?
                if gt_predicate.intersection(set(candidates)):
                    correct += 1

            results[f'top_{k}_accuracy'] = correct / total if total > 0 else 0.0
        return results

def mean_reciprocal_rank(ground_truth_list, predictions):
        """
            Calculate Mean Reciprocal Rank
        """
        reciprocal_ranks = []
        found_count = 0
        total_count = 0
        total = len(predictions)
        for i, pred in enumerate(predictions):
            gt_predicate = ground_truth_list[i]   
            if not gt_predicate:  # Skip None ground truth
                total -= 1
                continue
            
            if isinstance(gt_predicate, str):
                gt_predicate = [gt_predicate]
                
            # Find rank of ground truth predicate
            rank = None
            if 'Top_n_candidates' in pred:
                for j, (idx, candidate_data) in enumerate(pred['Top_n_candidates'].items()):
                    if candidate_data['mapped_predicate'] in gt_predicate:
                        rank = j + 1  # 1-indexed rank
                        found_count += 1
                        break

            if rank is not None:
                reciprocal_ranks.append(1.0 / rank)
            else:
                reciprocal_ranks.append(0.0)  # Not found in rankings

        return {
            'mean_reciprocal_rank': np.mean(reciprocal_ranks) if reciprocal_ranks else 0.0,
            'found_in_rankings_ratio': found_count / total if total > 0 else 0.0
        }

def error_analysis_with_confusion_matrix(ground_truth_list, predictions):
    """Enhanced error analysis using sklearn confusion matrix"""
    
    valid_ground_truth = []
    valid_predictions = []
    error_examples = []
    
    for i, pred in enumerate(predictions):
        gt_predicate = ground_truth_list[i].lower()
        s = pred["subject"]
        o = pred["object"]
        pred_predicate = pred['top_choice']['predicate'].replace('biolink:', '')
        if gt_predicate == pred_predicate:
            valid_predictions.append(pred_predicate)
        else:
            error_case = {
                'predicted_predicate': pred_predicate,
                'ground_truth_predicate': gt_predicate,
                'predicate_correct': False,
                'subject': s,
                'object': o
            }
            error_examples.append(error_case)
        
    all_labels = sorted(list(set(valid_ground_truth + valid_predictions)))
    
    cm = confusion_matrix(valid_ground_truth, valid_predictions, labels=all_labels)
    
    class_report = classification_report(
        valid_ground_truth, 
        valid_predictions, 
        labels=all_labels,
        output_dict=True,
        zero_division=0
    )

    return {
        'confusion_matrix': cm.tolist(),
        'confusion_matrix_labels': all_labels,
        'classification_report': class_report,
        'error_examples': error_examples[:20],
        'total_errors': len([ex for ex in error_examples if not ex['predicate_correct']]),
        'total_predictions': len(error_examples)
    }

def plot_confusion_matrix(confusion_matrix, labels, title=" Predicate Confusion Matrix", figsize=(12, 10), strikethrough_zero_rows=True):
    """Plot confusion matrix using seaborn heatmap"""
    
    if isinstance(confusion_matrix, list):
        confusion_matrix = np.array(confusion_matrix)
    
    y_labels = labels.copy()
    
    if strikethrough_zero_rows:
        row_sums = confusion_matrix.sum(axis=1)
        for i, label in enumerate(labels):
            if row_sums[i] == 0:
                y_labels[i] = f"~~{label}~~"
    
    plt.figure(figsize=figsize)
    
    sns.heatmap(
        confusion_matrix, 
        annot=True, 
        fmt='d', 
        # cmap='Blues',
        xticklabels=labels,
        yticklabels=y_labels,
        cbar_kws={'label': 'Count'}
    )
    
    plt.title(title)
    plt.xlabel('Predicted Predicate')
    plt.ylabel('True Predicate')
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    
    return plt.gcf()

def analyze_confusion_matrix(cm, labels):
    """Analyze confusion matrix to extract insights"""
    
    if isinstance(cm, list):
        cm = np.array(cm)
    
    insights = {}
    
    row_sums = cm.sum(axis=1)
    
    per_class_accuracy = {}
    labels_in_ground_truth = []
    labels_only_predicted = []
    
    for i, label in enumerate(labels):
        if row_sums[i] > 0:
            per_class_accuracy[label] = cm[i, i] / row_sums[i]
            labels_in_ground_truth.append(label)
        else:
            labels_only_predicted.append(label)
    
    confused_pairs = []
    for i in range(len(labels)):
        for j in range(len(labels)):
            if i != j and cm[i, j] > 0:
                confused_pairs.append({
                    'true_label': labels[i],
                    'predicted_label': labels[j],
                    'count': int(cm[i, j]),
                    'percentage_of_true_class': cm[i, j] / cm[i].sum() if cm[i].sum() > 0 else 0
                })
    
    confused_pairs.sort(key=lambda x: x['count'], reverse=True)
    
    insights['per_class_accuracy'] = per_class_accuracy
    insights['most_confused_pairs'] = confused_pairs[:10]
    insights['overall_accuracy'] = np.trace(cm) / np.sum(cm) if np.sum(cm) > 0 else 0
    insights['labels_in_ground_truth'] = labels_in_ground_truth
    insights['labels_only_predicted'] = labels_only_predicted
    insights['prediction_distribution'] = {label: int(cm[:, i].sum()) for i, label in enumerate(labels)}
    insights['ground_truth_distribution'] = {label: int(cm[i, :].sum()) for i, label in enumerate(labels) if cm[i, :].sum() > 0}
    
    return insights

def comprehensive_evaluation_enhanced(input_triples_with_ground_truth, output_results, selector, analysis_mode = "set"):   
    excluded = 0
    results = {}
    indices_to_remove = []

    output_results= [r for r in output_results 
        if len(r["subject"]) > 1 and len(r["object"]) > 1
    ]
    for i in range(len(output_results)):
        if output_results[i]['top_choice']['selector'] != selector:
            indices_to_remove.append(i)
            excluded += 1
    

    for i in reversed(indices_to_remove):
        input_triples_with_ground_truth.pop(i)
        output_results.pop(i)

    print(f"{excluded} triples excluded for evaluation result")

    if analysis_mode =="set":
        ground_truth_list = [inp["ground_truth_set"] for inp in input_triples_with_ground_truth]
    else:
        ground_truth_list = [inp["ground_truth"] for inp in input_triples_with_ground_truth]
  
    
    results['exact_match'], mismatched_indexes = exact_match_accuracy(ground_truth_list, output_results, selector)
    if isinstance(ground_truth_list[0], str):
        results['error_analysis'] = error_analysis_with_confusion_matrix(ground_truth_list, output_results)
        cm = results['error_analysis']['confusion_matrix']
        labels = results['error_analysis']['confusion_matrix_labels']
        results['confusion_matrix_insights'] = analyze_confusion_matrix(cm, labels)
    results['top_k_accuracy'] = top_k_accuracy(ground_truth_list, output_results)
    results['ranking_quality'] = mean_reciprocal_rank(ground_truth_list, output_results)   
    return results

def compare_experimental_triples_count_based_enhanced(input_triples_with_ground_truth, output_file, selector, analysis_mode = "set"):
    """Enhanced version of your comparison function"""
    with open(input_triples_with_ground_truth, "r") as f:
        input_with_original_categories = json.load(f)

    with open(output_file, 'r') as f:
        output_result = json.load(f)
    
    output_result = output_result["results"]

    results = comprehensive_evaluation_enhanced(
        input_with_original_categories, 
        output_result, selector, analysis_mode = analysis_mode
    )
    
    if 'error_analysis' in results:
        cm = results['error_analysis']['confusion_matrix']
        labels = results['error_analysis']['confusion_matrix_labels']
        
        fig = plot_confusion_matrix(cm, labels, strikethrough_zero_rows=True, title= "Predicate Confusion matrix")
        fig.savefig('predicate_confusion_matrix.png', dpi=300, bbox_inches='tight')
        plt.show()
        
        insights = results['confusion_matrix_insights']
        print(f"\nOverall Accuracy: {insights['overall_accuracy']:.3f}")
        
        print(f"\nLabels in ground truth: {len(insights['labels_in_ground_truth'])}")
        print(f"Labels only predicted (never in ground truth): {len(insights['labels_only_predicted'])}")
        if insights['labels_only_predicted']:
            print(f"  Predicted-only labels: {insights['labels_only_predicted']}")
        
        print("\nPer-class Accuracy (for labels with ground truth):")
        for label in insights['labels_in_ground_truth']:
            acc = insights['per_class_accuracy'][label]
            print(f"  {label}: {acc:.3f}")
        
        print("\nMost Frequent Confusions:")
        for confusion in insights['most_confused_pairs'][:5]:
            print(f"  {confusion['true_label']} -> {confusion['predicted_label']}: "
                  f"{confusion['count']} times ({confusion['percentage_of_true_class']:.1%} of true class)")
    
    return results

In [3]:
selector = "medgemma-27b-Q4"
analysis_mode = "set"
print("**********768")
print(compare_experimental_triples_count_based_enhanced("newest_chemprot_test_file_with_gt.json", "outputs1/chemprot_protocol_results_768_dim_noSapBert.json", selector=selector, analysis_mode=analysis_mode))
print()

print("**********768plus")
print(compare_experimental_triples_count_based_enhanced("newest_chemprot_test_file_with_gt.json", "outputs1/chemprot_protocol_results_768_dim.json", selector=selector, analysis_mode=analysis_mode))
print()

print("**********1024")
print(compare_experimental_triples_count_based_enhanced("newest_chemprot_test_file_with_gt.json", "outputs1/chemprot_protocol_results_1024_dim_noSapBert.json", selector=selector, analysis_mode=analysis_mode))
print()

print("**********1024plus")
print(compare_experimental_triples_count_based_enhanced("newest_chemprot_test_file_with_gt.json", "outputs1/chemprot_protocol_results_1024_dim.json", selector = selector, analysis_mode=analysis_mode))
print()

print("**********Baseline")
print(compare_experimental_triples_count_based_enhanced("newest_chemprot_test_file_with_gt.json", "outputs1/base_llm_on_chemprot_results_medgemma:27b.json", selector=selector, analysis_mode=analysis_mode))
print()

**********768
59 triples excluded for evaluation result
total instances evaluated for exact match: 636
{'exact_match': 0.4261006289308176, 'top_k_accuracy': {'top_1_accuracy': 0.5518867924528302, 'top_3_accuracy': 0.7012578616352201, 'top_5_accuracy': 0.7610062893081762, 'top_10_accuracy': 0.7751572327044025}, 'ranking_quality': {'mean_reciprocal_rank': 0.6323637316561844, 'found_in_rankings_ratio': 0.7751572327044025}}

**********768plus
15 triples excluded for evaluation result
total instances evaluated for exact match: 680
{'exact_match': 0.4573529411764706, 'top_k_accuracy': {'top_1_accuracy': 0.08676470588235294, 'top_3_accuracy': 0.2911764705882353, 'top_5_accuracy': 0.6397058823529411, 'top_10_accuracy': 0.9147058823529411}, 'ranking_quality': {'mean_reciprocal_rank': 0.28056547619047617, 'found_in_rankings_ratio': 0.9147058823529411}}

**********1024
33 triples excluded for evaluation result
total instances evaluated for exact match: 662
{'exact_match': 0.37462235649546827, 'to

In [4]:
def misses_analysis(input_triples_with_ground_truth_file, output_file):
    missed_rows = []
    possible = ['downregulator', 'regulator', 'upregulator', 'part of', 'has part', 'substrate', 'cofactor', 'antagonist', 'agonist']
    woke = []

    with open(input_triples_with_ground_truth_file, 'r') as f:
        input_triples_with_ground_truth = json.load(f)

    with open(output_file, 'r') as f:
        output_results = json.load(f)


    output_results = output_results["results"]
    for i in range(len(output_results)):
        r = output_results[i]
        postchoice = r['top_choice']['predicate']
        top1 = r.get('Top_n_candidates',{}).get('0', {}).get('mapped_predicate', '')
        gtset = input_triples_with_ground_truth[i]['ground_truth_set']
        if top1 in gtset and postchoice not in gtset:
            missed_rows.append({
                        "index": i,
                        "groundtruth": list(gtset),
                        "top1": top1,
                        "postchoice": postchoice,
                        "orignal_object": input_triples_with_ground_truth[i]["object"],
                        "orignal_subject":input_triples_with_ground_truth[i]["subject"],
                        "orignal_relationship":input_triples_with_ground_truth[i]["relationship"],
                        "abstract": input_triples_with_ground_truth[i]["abstract"]
                    })
        if postchoice not in possible:
            woke.append(i)
    
    # print(woke)
    pf = pd.DataFrame(missed_rows)
    pf.to_csv("missed_top1_df.csv", index = 0)
    
misses_analysis('newest_chemprot_test_file_with_gt.json', 'outputs1/chemprot_protocol_results_768_dim_noSapBert.json')

df = pd.read_csv('missed_top1_df.csv')

# complete error pattern distribution
# error_patterns = df.groupby(['groundtruth', 'postchoice']).size().reset_index(name='count')
# error_patterns['percentage'] = (error_patterns['count'] / 146 * 100).round(1)
# error_patterns = error_patterns.sort_values('count', ascending=False)

# print(error_patterns.to_string())

# complete pattern with original free-text relationships
error_patterns = df.groupby(['orignal_relationship', 'groundtruth', 'postchoice']).size().reset_index(name='count')
error_patterns['percentage'] = (error_patterns['count'] / len(df) * 100).round(1)
error_patterns = error_patterns.sort_values('count', ascending=False)
print(error_patterns.to_string())

# grouped = error_patterns.groupby(['groundtruth', 'postchoice']).agg({
#     'orignal_relationship': lambda x: list(x),  # Collect all free-text variants
#     'count': 'sum'  # Sum the counts
# }).reset_index()

# grouped['percentage'] = (grouped['count'] / len(df) * 100).round(1)
# grouped = grouped.sort_values('count', ascending=False)


# grouped['example_relationships'] = grouped['orignal_relationship'].apply(
#     lambda x: ', '.join(f'"{item}"' for item in x)
# )

# print(grouped[['groundtruth', 'postchoice', 'example_relationships', 'count', 'percentage']].to_string())

                                    orignal_relationship              groundtruth              postchoice  count  percentage
22                                              inhibits        ['downregulator']                inhibits     68        46.6
4                                                 blocks        ['downregulator']              antagonist      8         5.5
20                                              inhibits        ['downregulator']              antagonist      6         4.1
0                                              activates          ['upregulator']               activates      4         2.7
26                              inhibits the activity of        ['downregulator']                inhibits      4         2.7
1                                              activates          ['upregulator']                 agonist      4         2.7
5                                                 blocks        ['downregulator']                  blocks      3         2.1


In [5]:
 # ****768*****
# 695 695 695
# total instances evaluated for exact match: 571
# total instances evaluated for accuracy@k and MRR: 695
# max Top_n size: 7
# example mismatches: [{'predicted': 'upregulator', 'valid_ground_truth': ['substrate'], 'subject': 'TES', 'object': 'xanthine oxidase', 'top_candidates_sorted_by_score': [('upregulator', 0.684), ('agonist', 0.655), ('substrate', 0.651)]}, {'predicted': 'upregulator', 'valid_ground_truth': ['regulator'], 'subject': 'DAGs', 'object': 'PTP1B', 'top_candidates_sorted_by_score': [('upregulator', 0.661), ('modulator', 0.563), ('downregulator', 0.561)]}]
# 
# {'set_match_accuracy': 0.46409807355516636, 'top_1_accuracy': 0.5280575539568345, 'top_3_accuracy': 0.6863309352517986, 'top_5_accuracy': 0.7438848920863309, 'top_10_accuracy': 0.7553956834532374, 'MRR': 0.611558752997602}
# 
# Consistency check: 290 instances where top_choice doesn't match highest scoring candidate